# Clase 24 — Bonus track: cómo funciona RAG por dentro

**Diplomado IA Aplicada al Diseño · UDD · 2026** · Prof. Darío Osorio

Este es el **último notebook de Colab del curso**. Es opcional pero recomendado. Muestra en 20 minutos cómo funciona por dentro el patrón que usa NotebookLM (y muchas otras herramientas).

Después de esta clase pasamos 100% a herramientas SaaS.

## Objetivo
Construir un mini-RAG con 3 documentos ficticios y hacerle preguntas.

In [1]:
!pip install -q sentence-transformers numpy
print("Listo.")

Listo.


## Paso 1 — Nuestros "documentos"

En NotebookLM cargas PDFs. Acá simulamos con strings.

In [2]:
documentos = [
    """Entrevista 1 - Maria, 45 anos, Concepcion.
    Lo que mas me molesta de la app del banco es que me cierra la sesion cada 5 minutos.
    Para hacer una transferencia entro 4 veces.""",

    """Entrevista 2 - Felipe, 28, Antofagasta. Diseñador.
    La interfaz visualmente esta ok pero los iconos son confusos.
    El de tarjetas parece de configuracion.""",

    """Entrevista 3 - Claudia, 67, Valparaiso.
    No entiendo bien la app. La letra es muy chica.
    Cuando me llaman del banco no se si es real. Tengo miedo a apretar algo malo.""",
]
print(f"{len(documentos)} documentos cargados.")

3 documentos cargados.


## Paso 2 — Convertir documentos a embeddings

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

modelo = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = modelo.encode(documentos)
print(f"Shape: {embeddings.shape}")
print("Cada documento es ahora un vector de 384 dimensiones.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape: (3, 384)
Cada documento es ahora un vector de 384 dimensiones.


## Paso 3 — Función de búsqueda semántica

Le damos una pregunta, ella busca el documento más parecido.

In [4]:
def buscar(pregunta, top_k=1):
    p_emb = modelo.encode([pregunta])
    sims = np.dot(embeddings, p_emb.T).flatten()
    sims = sims / (np.linalg.norm(embeddings, axis=1) * np.linalg.norm(p_emb))
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(documentos[i], float(sims[i])) for i in top_idx]

preguntas = [
    "que problemas de seguridad mencionan?",
    "hay algo sobre los iconos?",
    "que dijeron los adultos mayores?",
]

for q in preguntas:
    print(f"P: {q}")
    for doc, sim in buscar(q, top_k=1):
        print(f"  Similitud: {sim:.2f}")
        print(f"  Documento: {doc[:150]}...")
    print()

P: que problemas de seguridad mencionan?
  Similitud: 0.51
  Documento: Entrevista 2 - Felipe, 28, Antofagasta. Diseñador.
    La interfaz visualmente esta ok pero los iconos son confusos.
    El de tarjetas parece de conf...

P: hay algo sobre los iconos?
  Similitud: 0.54
  Documento: Entrevista 2 - Felipe, 28, Antofagasta. Diseñador.
    La interfaz visualmente esta ok pero los iconos son confusos.
    El de tarjetas parece de conf...

P: que dijeron los adultos mayores?
  Similitud: 0.55
  Documento: Entrevista 2 - Felipe, 28, Antofagasta. Diseñador.
    La interfaz visualmente esta ok pero los iconos son confusos.
    El de tarjetas parece de conf...



## Cierre

Esto es exactamente lo que hace NotebookLM (más pulido y con Gemini razonando encima). Ahora que viste el mecanismo, ya sabes por qué:

- Cuando la respuesta "no está en las fuentes", el sistema puede decir "no sé".
- Las citas son literales (vienen del documento recuperado).
- La calidad depende del corte (chunking) y de qué tan buenas son tus fuentes.

**No hace falta que ejecutes esto en tu proyecto** — usa NotebookLM directamente. Este notebook es solo para entender qué pasa por debajo.